In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Master/LTAINC/lightweight-medical-model')
DATASET_DIR = PROJECT_DIR / 'data/busi'
OUTPUTS_ROOT = PROJECT_DIR / 'outputs'
RESULTS_ROOT = PROJECT_DIR / 'augmentation-comparison-results'

assert DATASET_DIR.is_dir(), f'Không tìm thấy BUSI: {DATASET_DIR}'
assert (PROJECT_DIR / 'run_augmentation_comparison.py').is_file(), 'Thiếu comparison runner'
%cd {PROJECT_DIR}

# Augmentation comparison — MK-MNet width 0.25

Thiết kế dựa trên `docs/ref/data-augmentation.pdf`: no augmentation, single augmentation, fixed legacy policy, TrivialAugment và speckle extension.

In [ ]:
POLICIES = [
    'none',
    'rotate', 'translate_y', 'scale',
    'horizontal_flip', 'vertical_flip',
    'brightness', 'contrast', 'gaussian_noise', 'elastic',
    'legacy', 'trivial_all_1', 'trivial_all_3',
    'speckle_noise',
]
SEEDS = [42, 123, 2026]
print(f'{len(POLICIES)} policies × {len(SEEDS)} seeds = {len(POLICIES) * len(SEEDS)} training runs')

## Kiểm tra trực quan image-mask synchronization trước khi train

In [ ]:
policy_args = ' '.join(POLICIES)
!python preview_augmentation_policies.py \
  --dataset-dir "$DATASET_DIR" \
  --output-dir "$RESULTS_ROOT/previews" \
  --policies $policy_args \
  --samples 4

In [ ]:
from IPython.display import Image, display

for path in sorted((RESULTS_ROOT / 'previews').glob('*.png')):
    print(path.stem)
    display(Image(filename=str(path)))

## Chạy comparison

Runner dùng 100 epoch, no early stopping, oversampling cố định và tự resume các run đã hoàn thành.

In [ ]:
seed_args = ' '.join(map(str, SEEDS))
!python run_augmentation_comparison.py \
  --dataset-dir "$DATASET_DIR" \
  --outputs-root "$OUTPUTS_ROOT" \
  --results-root "$RESULTS_ROOT" \
  --policies $policy_args \
  --seeds $seed_args \
  --epochs 100 \
  --batch-size 16 \
  --resume

## Kiểm tra trạng thái và mean ± std

In [ ]:
import pandas as pd
from IPython.display import display

status = pd.read_csv(RESULTS_ROOT / 'comparison_status.csv')
display(status)
failed = status[(status.training != 'completed') | (status.evaluation != 'completed')]
print(f'Failed runs: {len(failed)}')

In [ ]:
summary = pd.read_csv(RESULTS_ROOT / 'comparison_summary.csv')
display(summary[[
    'policy', 'runs',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'macro_f1_mean', 'macro_f1_std',
    'malignant_sensitivity_mean', 'malignant_sensitivity_std',
    'malignant_specificity_mean', 'malignant_specificity_std',
    'dice_lesion_mean', 'dice_lesion_std',
]].sort_values('balanced_accuracy_mean', ascending=False))